In [75]:
import pandas as pd
import numpy as np

In [76]:
filename ="exploration_geology_messy_data.csv"

In [77]:
#Load the data 
geology = pd.read_csv(filename)

#Show 5 random rows of results
geology.sample(5)

,Hole_ID,From_m,To_m,Au_Result,Au_Unit,Lithology,Sample_ID
99,BH-100,16.18,12.80,-2.478,ppb,dolerit,SMP100
26,NaN,29.95,145.95,-0.378,g/t,baslt,SMP27
172,BH-173,76.81,75.81,-0.553,GT,Grnt,SMP173
124,BH-125,34.19,143.23,-0.503,g/t,baslt,SMP125
119,BH-120,80.90,37.54,1.918,g/t,baslt,SMP120


In [78]:
#How many rows of results are in the dataframe
geology.shape[0]
 #About 200 rows of results

200

In [79]:
#Print of the datatpes of each column

geology.dtypes #All datatypes seems to be well aligned 

Hole_ID          str
From_m       float64
To_m         float64
Au_Result    float64
Au_Unit          str
Lithology        str
Sample_ID        str
dtype: object

In [80]:
geology.head(10)

,Hole_ID,From_m,To_m,Au_Result,Au_Unit,Lithology,Sample_ID
0,BH-001,56.18,96.30,-2.505,GT,Grnt,SMP1
1,BH-002,142.61,12.62,-1.221,ppb,Grnt,SMP2
2,BH-003,109.80,24.24,1.882,GT,Granite,SMP3
3,BH-004,89.80,134.78,0.890,NaN,gran ite,SMP4
4,BH-005,23.40,90.96,1.524,NaN,Dolerite,SMP5
5,BH-006,23.40,1.38,1.070,g/t,NaN,SMP6
6,bh-007,8.71,15.22,1.297,NaN,Basalt,SMP7
7,BH-008,129.93,99.53,1.795,NaN,gran ite,SMP8
8,BH-009,90.17,0.76,0.382,ppb,Dolerite,SMP9
9,BH-010,106.21,24.12,3.292,NaN,BAZALT,SMP10


In [81]:
#Define function to transform and standardize Hole_ID

def Hole_ID(geology):
    geology['Hole_ID'] = (geology['Hole_ID']
        .str.strip()
        .str.upper()
        .str.replace(r"([A-Z]{2})[\s_-]*(\d+)", r"\1-\2")
                         )
    return geology
#Define a function to auto-fill drillholes that are missing)

def drillhole_fill (geology):
    geology['Hole_ID'] = (geology['Hole_ID']
                          .str.extract(r'(\d+)')
                          .astype(float)
                         )
    missing = geology['Hole_ID'].interpolate().astype(int)

    geology['Hole_ID'] = 'BP-' + missing.astype(str).str.zfill(3)
    return geology

#Define the new interval function, swap the from and to, and insert the new interval 

def Interval(geology): 
    #Mask the rows with a great From_m than To_m
    mask = geology['From_m'] > geology['To_m']
    
    #Swap the masked rows
    geology.loc[mask, ['From_m', 'To_m']] = geology.loc[mask, ['To_m', 'From_m']].values
    
    #Define the position of new column 
    index = geology.columns.get_loc('To_m') + 1
    
    #Define the calculation to be used
    interval = geology['To_m'] - geology['From_m']

    #insert the new columns into the dataframe
    geology.insert(index,'Interval_m', interval)

    #Print out the dataframe with a negative interval

    print("Total number of rows with negative interval:",(geology['Interval_m']<0).sum())

    return geology

#Define a function to clean and transform the Au Results removing all the Negative and converting the Au_Units to g/t

def Au_Values (geology):
    #Start by removing the negative values into np.nan
    geology['Au_Result'] = geology['Au_Result'].apply(
    lambda x: np.nan if x < 0 else x
)
    
    #Convert ppb to g/t
    #Start by standardizing and propering the units 
    geology['Au_Unit'] = (
        geology['Au_Unit']
        .str.strip()
        .str.lower()
    )

    #Start by filling the NULL Au_Unit with g/t

    geology['Au_Unit']=geology['Au_Unit'].fillna('g/t')

    #Map the Au units
    Au_map = {'g/t' : 1,
              'gt': 1,
              'ppm': 1, 
              'ppb': 1/1000}
    #Apply the mapped data over the Au result

    geology['Au_Result'] =(
        geology['Au_Result']
        *geology['Au_Unit'].map(Au_map)
    )

    #Then now change the Au units into g/t

    geology['Au_Unit'] = 'g/t'

    return geology

def Lithology (geology):
    #Proper the geological lithologies
    geology['Lithology'] = geology['Lithology'].str.strip().fillna('Unknown')

    #Replace the names
    rock_names = {'Grnt' : 'Granite',
                  'Granite': 'Granite',
                  'gran ite':'Granite',
                  'Dolerite':'Dolerite',
                  'Basalt':'Basalt',
                  'BAZALT': 'Basalt',
                  'dolerit':'Dolerite',
                  'baslt':'Basalt'}
    geology['Lithology'] = geology['Lithology'].map(rock_names)

    return geology


def Sample_ID (geology):
    sample_clean = geology['Sample_ID'].str.extract(r"(\d+)").astype(float)

    sample_clean = sample_clean.interpolate().astype(int)
    geology['Sample_ID'] = 'SMP-' + sample_clean.astype(str)

    return geology


In [82]:
exploration.columns

Index(['Hole_ID', 'From_m', 'To_m', 'Interval_m', 'Au_Result', 'Au_Unit',
       'Lithology', 'Sample_ID'],
      dtype='str')

In [83]:
exploration = (
    geology
        .pipe(Hole_ID)
        .pipe(drillhole_fill)
        .pipe(Interval)
        .pipe (Au_Values)
        .pipe(Lithology)
        .pipe(Sample_ID)
)
      


Total number of rows with negative interval: 0


In [86]:
#Print out the top 20 rows of the cleaned data
exploration.head(20)

,Hole_ID,From_m,To_m,Interval_m,Au_Result,Au_Unit,Lithology,Sample_ID
0,BP-001,56.18,96.30,40.12,NaN,g/t,Granite,SMP-1
1,BP-002,12.62,142.61,129.99,NaN,g/t,Granite,SMP-2
2,BP-003,24.24,109.80,85.56,1.882000,g/t,Granite,SMP-3
3,BP-004,89.80,134.78,44.98,0.890000,g/t,Granite,SMP-4
4,BP-005,23.40,90.96,67.56,1.524000,g/t,Dolerite,SMP-5
5,BP-006,1.38,23.40,22.02,1.070000,g/t,NaN,SMP-6
6,BP-007,8.71,15.22,6.51,1.297000,g/t,Basalt,SMP-7
7,BP-008,99.53,129.93,30.40,1.795000,g/t,Granite,SMP-8
8,BP-009,0.76,90.17,89.41,0.000382,g/t,Dolerite,SMP-9
9,BP-010,24.12,106.21,82.09,3.292000,g/t,Basalt,SMP-10


In [ ]:
#THIS WAS COMPILED BY QOLI AS HE HARNESS SKILLS IN DATA EXPLORATION AND ANALYSIS